## C++ acceleration (pybind11)

The next two cells build the extension `ambs_core` (C++ port of the AMBS simulation), which runs the grid search and the 10 000-trajectory evaluation. 

In [ ]:
%%writefile ambs_core.cpp
// ambs_core.cpp
// C++ port of the AMBS simulation and grid search of AMBS_Heuristic_SCLSP.ipynb (DynaPlex-faithful),
// written to make the grid search fast.
// RNG: one std::mt19937_64 per trajectory, seeded from (base_seed, trajectory index). The streams are
// independent and reproducible but not bit-identical to the numpy ones of the Python version.

#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <vector>
#include <string>
#include <random>
#include <cmath>
#include <algorithm>
#include <limits>
#include <numeric>
#ifdef _OPENMP
#include <omp.h>
#endif

namespace py = pybind11;

// Instance
struct Instance {
    int K;
    std::vector<double> mu, sigma, h, b, k;
    std::vector<int> theta;
    int capacity;
    std::string corr_mode; // "", "pair_plus_one", "pair_same", "group_same"
    std::vector<std::pair<int,int>> corr_pairs;
    std::vector<std::vector<int>> corr_groups;

    std::vector<long long> Q;   // EOQ
    std::vector<double> C;      // EOQ cost
    std::vector<long long> lower, upper; // demand bounds

    void finalize() {
        Q.resize(K);
        C.resize(K);
        lower.resize(K);
        upper.resize(K);
        for (int i = 0; i < K; ++i) {
            Q[i] = (long long)std::ceil(std::sqrt(2.0 * mu[i] * k[i] / h[i]));
            if (Q[i] < 1) Q[i] = 1;
            C[i] = k[i] * mu[i] / (double)Q[i] + h[i] * (double)Q[i] / 2.0;
            long long lo = (long long)std::llround(mu[i] - sigma[i]);
            if (lo < 0) lo = 0;
            long long hi = (long long)std::llround(mu[i] + sigma[i]);
            lower[i] = lo;
            upper[i] = hi;
        }
    }
};

struct AMBSParams {
    double Bmin;
    double Hmax;
    int Zmax;
};

// Demand
static inline long long clip_ll(long long v, long long lo, long long hi) {
    return std::max(lo, std::min(v, hi));
}

static void sample_demand(const Instance& inst, std::mt19937_64& rng,
                           std::vector<long long>& d) {
    d.resize(inst.K);
    for (int i = 0; i < inst.K; ++i) {
        std::uniform_int_distribution<long long> dist(inst.lower[i], inst.upper[i]);
        d[i] = dist(rng);
    }
    if (inst.corr_mode == "pair_plus_one") {
        for (auto& pr : inst.corr_pairs) {
            int i = pr.first, j = pr.second;
            std::uniform_int_distribution<long long> dist(inst.lower[i], inst.upper[i]);
            long long base = dist(rng);
            d[i] = base;
            d[j] = clip_ll(base + 1, inst.lower[j], inst.upper[j]);
        }
    } else if (inst.corr_mode == "pair_same") {
        for (auto& pr : inst.corr_pairs) {
            int i = pr.first, j = pr.second;
            std::uniform_int_distribution<long long> dist(inst.lower[i], inst.upper[i]);
            long long base = dist(rng);
            d[i] = clip_ll(base, inst.lower[i], inst.upper[i]);
            d[j] = clip_ll(base, inst.lower[j], inst.upper[j]);
        }
    } else if (inst.corr_mode == "group_same") {
        for (auto& grp : inst.corr_groups) {
            if (grp.empty()) continue;
            int i0 = grp[0];
            std::uniform_int_distribution<long long> dist(inst.lower[i0], inst.upper[i0]);
            long long base = dist(rng);
            for (int j : grp) d[j] = clip_ll(base, inst.lower[j], inst.upper[j]);
        }
    }
}

static inline double expected_backlog(const Instance& inst, int i, double x) {
    long long lo = inst.lower[i], hi = inst.upper[i];
    long long n = hi - lo + 1;
    double p = 1.0 / (double)n;
    double total = 0.0;
    for (long long dd = lo; dd <= hi; ++dd) {
        double diff = (double)dd - x;
        if (diff > 0.0) total += p * diff;
    }
    return total;
}

// AMBS heuristic: production plan
static void determine_production_plan(const Instance& inst, const std::vector<long long>& inventory,
                                       int product_currently_set_up, int used_capacity,
                                       const AMBSParams& params,
                                       std::vector<long long>& planned_production,
                                       int& last_product_scheduled) {
    int K = inst.K;
    int capacity = inst.capacity;
    long long planned_capacity = used_capacity;
    planned_production.assign(K, 0);
    std::vector<int> new_setup_skus;
    bool last_product_determined = false;
    last_product_scheduled = -1;

    std::vector<double> ebo(K), ebo_exceeded(K);

    while (planned_capacity < capacity) {
        std::fill(ebo.begin(), ebo.end(), 0.0);
        std::fill(ebo_exceeded.begin(), ebo_exceeded.end(), 0.0);

        for (int i = 0; i < K; ++i) {
            bool in_new = std::find(new_setup_skus.begin(), new_setup_skus.end(), i) != new_setup_skus.end();
            bool eligible = in_new || (i == product_currently_set_up) ||
                             ((int)new_setup_skus.size() < params.Zmax);
            if (eligible) {
                ebo[i] = expected_backlog(inst, i, (double)(inventory[i] + planned_production[i]));
            }
            // params.Bmin is already the scaled threshold (xb * mean EOQ cost): no rescaling by C[i].
            double v = ebo[i] - params.Bmin;
            ebo_exceeded[i] = v > 0.0 ? v : 0.0;
        }

        int i_best = (int)(std::max_element(ebo_exceeded.begin(), ebo_exceeded.end()) - ebo_exceeded.begin());
        double best_val = ebo_exceeded[i_best];

        if (best_val > 0.0) {
            bool in_new = std::find(new_setup_skus.begin(), new_setup_skus.end(), i_best) != new_setup_skus.end();
            if (!in_new && i_best != product_currently_set_up) {
                new_setup_skus.push_back(i_best);
                planned_capacity += inst.theta[i_best];
                if (planned_capacity >= capacity) {
                    last_product_determined = true;
                    last_product_scheduled = i_best;
                }
            }
            if (planned_capacity < capacity) {
                planned_production[i_best] += 1;
                planned_capacity += 1;
            }
        } else {
            std::vector<int> produced_skus;
            produced_skus.push_back(product_currently_set_up);
            for (int s : new_setup_skus) produced_skus.push_back(s);

            if ((int)produced_skus.size() <= 2) {
                last_product_determined = true;
                last_product_scheduled = (produced_skus.size() == 1) ? produced_skus[0] : produced_skus[1];

                int i0 = produced_skus[0];
                while (planned_capacity < capacity) {
                    if ((double)(inventory[i0] + planned_production[i0] + 1) < params.Hmax * (double)inst.Q[i0]) {
                        planned_production[i0] += 1;
                        planned_capacity += 1;
                    } else {
                        break;
                    }
                }
            } else {
                std::vector<double> inv_ratio_new;
                for (int s : new_setup_skus)
                    inv_ratio_new.push_back((double)(planned_production[s] + inventory[s]) / (double)inst.Q[s]);
                int argmin_idx = (int)(std::min_element(inv_ratio_new.begin(), inv_ratio_new.end()) - inv_ratio_new.begin());
                last_product_scheduled = new_setup_skus[argmin_idx];
                last_product_determined = true;

                produced_skus.erase(std::remove(produced_skus.begin(), produced_skus.end(), last_product_scheduled),
                                     produced_skus.end());

                std::vector<double> inv_ratio;
                for (int s : produced_skus)
                    inv_ratio.push_back((double)(planned_production[s] + inventory[s]) / (double)inst.Q[s]);

                while (planned_capacity < capacity && !produced_skus.empty()) {
                    int j = (int)(std::min_element(inv_ratio.begin(), inv_ratio.end()) - inv_ratio.begin());
                    int s = produced_skus[j];
                    if ((double)(inventory[s] + planned_production[s] + 1) < params.Hmax * (double)inst.Q[s]) {
                        planned_production[s] += 1;
                        planned_capacity += 1;
                    } else {
                        produced_skus.erase(produced_skus.begin() + j);
                        inv_ratio.erase(inv_ratio.begin() + j);
                    }
                }
            }

            while (planned_capacity < capacity && last_product_determined) {
                int s = last_product_scheduled;
                if ((double)(inventory[s] + planned_production[s] + 1) < params.Hmax * (double)inst.Q[s]) {
                    planned_production[s] += 1;
                    planned_capacity += 1;
                } else {
                    planned_capacity = capacity; // force stop, mirrors reference
                }
            }
        }
    }

    if (!last_product_determined) {
        if (new_setup_skus.empty()) {
            last_product_scheduled = product_currently_set_up;
        } else {
            std::vector<double> inv_ratio_new;
            for (int s : new_setup_skus)
                inv_ratio_new.push_back((double)(planned_production[s] + inventory[s]) / (double)inst.Q[s]);
            int argmin_idx = (int)(std::min_element(inv_ratio_new.begin(), inv_ratio_new.end()) - inv_ratio_new.begin());
            last_product_scheduled = new_setup_skus[argmin_idx];
        }
    }
}

static void realize_production_plan(const Instance& inst, const std::vector<long long>& planned_production,
                                     int product_currently_set_up, int last_product_scheduled,
                                     int used_capacity,
                                     std::vector<long long>& actual, int& n_new_setups,
                                     double& setup_cost_total) {
    int K = inst.K;
    int capacity = inst.capacity;

    std::vector<int> order;
    if (planned_production[product_currently_set_up] > 0 && product_currently_set_up != last_product_scheduled) {
        order.push_back(product_currently_set_up);
    }
    for (int i = 0; i < K; ++i) {
        if (i == product_currently_set_up || i == last_product_scheduled) continue;
        if (planned_production[i] > 0) order.push_back(i);
    }
    if (planned_production[last_product_scheduled] > 0 || last_product_scheduled == product_currently_set_up) {
        order.push_back(last_product_scheduled);
    }

    actual.assign(K, 0);
    int capacity_used = used_capacity;
    n_new_setups = 0;
    setup_cost_total = 0.0;
    int current_setup = product_currently_set_up;

    for (int i : order) {
        int remaining = capacity - capacity_used;
        if (remaining <= 0) break;
        bool needs_setup = (i != current_setup);
        if (needs_setup) {
            remaining -= inst.theta[i];
            if (remaining <= 0) continue;
            n_new_setups += 1;
            setup_cost_total += inst.k[i];
            capacity_used += inst.theta[i];
            current_setup = i;
        }
        long long produced_qty = std::min(planned_production[i], (long long)(capacity - capacity_used));
        if (produced_qty <= 0) continue;
        actual[i] = produced_qty;
        capacity_used += (int)produced_qty;
    }
}

// Trajectory simulation
static double simulate_trajectory(const Instance& inst, const AMBSParams& params,
                                   std::mt19937_64& rng, int warmup_periods, int periods_measured) {
    int K = inst.K;
    std::vector<long long> inventory(K);
    for (int i = 0; i < K; ++i)
        inventory[i] = (long long)std::ceil(inst.mu[i] + 1.282 * inst.sigma[i]);

    int product_currently_set_up = 0;
    int used_capacity = 0;

    int total_periods = warmup_periods + periods_measured;
    double cumulative_cost = 0.0;
    double checkpoint_at_warmup = 0.0;

    std::vector<long long> planned_production, actual_production, demand_draw;

    for (int t = 0; t < total_periods; ++t) {
        int last_product;
        determine_production_plan(inst, inventory, product_currently_set_up, used_capacity,
                                   params, planned_production, last_product);

        int n_new_setups; double setup_cost_period;
        realize_production_plan(inst, planned_production, product_currently_set_up, last_product,
                                 used_capacity, actual_production, n_new_setups, setup_cost_period);

        for (int i = 0; i < K; ++i) inventory[i] += actual_production[i];

        sample_demand(inst, rng, demand_draw);
        for (int i = 0; i < K; ++i) inventory[i] -= demand_draw[i];

        double holding_cost = 0.0, backorder_cost = 0.0;
        for (int i = 0; i < K; ++i) {
            if (inventory[i] > 0) holding_cost += inst.h[i] * (double)inventory[i];
            else backorder_cost += inst.b[i] * (double)(-inventory[i]);
        }
        double period_cost = holding_cost + backorder_cost + setup_cost_period;
        cumulative_cost += period_cost;

        product_currently_set_up = last_product;
        used_capacity = 0;

        if (t == warmup_periods - 1) checkpoint_at_warmup = cumulative_cost;
    }

    return (cumulative_cost - checkpoint_at_warmup) / (double)periods_measured;
}

static std::mt19937_64 make_trajectory_rng(long long base_seed, long long trajectory_index) {
    std::seed_seq seq{
        (uint32_t)(base_seed & 0xffffffffULL),
        (uint32_t)((base_seed >> 32) & 0xffffffffULL),
        (uint32_t)(trajectory_index & 0xffffffffULL),
        (uint32_t)((trajectory_index >> 32) & 0xffffffffULL)
    };
    return std::mt19937_64(seq);
}

static void evaluate_ambs(const Instance& inst, const AMBSParams& params, int n_trajectories,
                           long long base_seed, int warmup_periods, int periods_measured,
                           double& mean_out, double& se_out) {
    std::vector<double> costs(n_trajectories);
    for (int i = 0; i < n_trajectories; ++i) {
        std::mt19937_64 rng = make_trajectory_rng(base_seed, i);
        costs[i] = simulate_trajectory(inst, params, rng, warmup_periods, periods_measured);
    }
    double mean = std::accumulate(costs.begin(), costs.end(), 0.0) / n_trajectories;
    double sq = 0.0;
    for (double c : costs) sq += (c - mean) * (c - mean);
    double sd = (n_trajectories > 1) ? std::sqrt(sq / (n_trajectories - 1)) : 0.0;
    mean_out = mean;
    se_out = sd / std::sqrt((double)n_trajectories);
}

// Instance builder from Python args
static Instance build_instance(int K, std::vector<double> mu, std::vector<double> sigma,
                                std::vector<double> h, std::vector<double> b, std::vector<double> k,
                                std::vector<int> theta, int capacity, std::string corr_mode,
                                std::vector<std::pair<int,int>> corr_pairs,
                                std::vector<std::vector<int>> corr_groups) {
    Instance inst;
    inst.K = K;
    inst.mu = std::move(mu);
    inst.sigma = std::move(sigma);
    inst.h = std::move(h);
    inst.b = std::move(b);
    inst.k = std::move(k);
    inst.theta = std::move(theta);
    inst.capacity = capacity;
    inst.corr_mode = std::move(corr_mode);
    inst.corr_pairs = std::move(corr_pairs);
    inst.corr_groups = std::move(corr_groups);
    inst.finalize();
    return inst;
}

// pybind11-exposed functions
py::tuple py_evaluate(int K, std::vector<double> mu, std::vector<double> sigma,
                       std::vector<double> h, std::vector<double> b, std::vector<double> k,
                       std::vector<int> theta, int capacity, std::string corr_mode,
                       std::vector<std::pair<int,int>> corr_pairs,
                       std::vector<std::vector<int>> corr_groups,
                       double Bmin, double Hmax, int Zmax,
                       int n_trajectories, long long base_seed,
                       int warmup_periods, int periods_measured, bool per_product) {
    Instance inst = build_instance(K, mu, sigma, h, b, k, theta, capacity, corr_mode, corr_pairs, corr_groups);
    AMBSParams params{Bmin, Hmax, Zmax};
    double mean, se;
    evaluate_ambs(inst, params, n_trajectories, base_seed, warmup_periods, periods_measured, mean, se);
    if (per_product) { mean /= K; se /= K; }
    return py::make_tuple(mean, se);
}

py::dict py_grid_search(int K, std::vector<double> mu, std::vector<double> sigma,
                         std::vector<double> h, std::vector<double> b, std::vector<double> k,
                         std::vector<int> theta, int capacity, std::string corr_mode,
                         std::vector<std::pair<int,int>> corr_pairs,
                         std::vector<std::vector<int>> corr_groups,
                         std::vector<double> xb_grid, std::vector<double> xh_grid,
                         std::vector<int> z_grid,
                         int n_trajectories, long long base_seed,
                         int warmup_periods, int periods_measured, int n_threads) {
    Instance inst = build_instance(K, mu, sigma, h, b, k, theta, capacity, corr_mode, corr_pairs, corr_groups);
    double Bmin_scale = std::accumulate(inst.C.begin(), inst.C.end(), 0.0) / (double)K; // mean(C)

    struct Combo { double xb, xh; int z; };
    std::vector<Combo> combos;
    combos.reserve(xb_grid.size() * xh_grid.size() * z_grid.size());
    for (double xb : xb_grid)
        for (double xh : xh_grid)
            for (int z : z_grid)
                combos.push_back({xb, xh, z});

    int n = (int)combos.size();
    std::vector<double> results(n);

#ifdef _OPENMP
    if (n_threads > 0) omp_set_num_threads(n_threads);
    #pragma omp parallel for schedule(dynamic)
#endif
    for (int idx = 0; idx < n; ++idx) {
        AMBSParams params{combos[idx].xb * Bmin_scale, combos[idx].xh, combos[idx].z};
        double mean, se;
        evaluate_ambs(inst, params, n_trajectories, base_seed, warmup_periods, periods_measured, mean, se);
        results[idx] = mean;
    }

    int best_idx = (int)(std::min_element(results.begin(), results.end()) - results.begin());

    py::list all_results;
    for (int idx = 0; idx < n; ++idx) {
        all_results.append(py::make_tuple(combos[idx].xb, combos[idx].xh, combos[idx].z, results[idx]));
    }

    py::dict out;
    out["Bmin"] = combos[best_idx].xb * Bmin_scale;
    out["Hmax"] = combos[best_idx].xh;
    out["Zmax"] = combos[best_idx].z;
    out["cost"] = results[best_idx];
    out["all_results"] = all_results;
    return out;
}

PYBIND11_MODULE(ambs_core, m) {
    m.doc() = "Fast C++ core for the AMBS heuristic (grid search + evaluation)";
    m.def("evaluate", &py_evaluate, "Evaluate AMBS for a fixed (Bmin, Hmax, Zmax)",
          py::arg("K"), py::arg("mu"), py::arg("sigma"), py::arg("h"), py::arg("b"), py::arg("k"),
          py::arg("theta"), py::arg("capacity"), py::arg("corr_mode"), py::arg("corr_pairs"),
          py::arg("corr_groups"), py::arg("Bmin"), py::arg("Hmax"), py::arg("Zmax"),
          py::arg("n_trajectories"), py::arg("base_seed"), py::arg("warmup_periods"),
          py::arg("periods_measured"), py::arg("per_product") = false);

    m.def("grid_search", &py_grid_search, "Grid search over (xb, xh, z) thresholds, parallel over combos",
          py::arg("K"), py::arg("mu"), py::arg("sigma"), py::arg("h"), py::arg("b"), py::arg("k"),
          py::arg("theta"), py::arg("capacity"), py::arg("corr_mode"), py::arg("corr_pairs"),
          py::arg("corr_groups"), py::arg("xb_grid"), py::arg("xh_grid"), py::arg("z_grid"),
          py::arg("n_trajectories"), py::arg("base_seed"), py::arg("warmup_periods"),
          py::arg("periods_measured"), py::arg("n_threads") = 0);
}


Writing ambs_core.cpp


In [ ]:
import subprocess, sys, sysconfig, os

def _pip_install(pkg):
    """Install with pip; retry with --break-system-packages on externally-managed (PEP 668) environments."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                        capture_output=True, text=True)
    if r.returncode != 0 and "externally-managed-environment" in (r.stderr or ""):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                             "--break-system-packages", pkg],
                            capture_output=True, text=True)
    return r

pip_result = _pip_install("pybind11")
if pip_result.returncode != 0:
    print("--- pip stdout ---"); print(pip_result.stdout)
    print("--- pip stderr ---"); print(pip_result.stderr)
    raise RuntimeError("pip install pybind11 failed. See output above.")

# Build in the kernel's working directory so the .so can be imported.
build_dir = os.getcwd()
print("Building in:", build_dir)

py_include = sysconfig.get_path("include")
pybind_include = subprocess.run(
    [sys.executable, "-c", "import pybind11; print(pybind11.get_include())"],
    capture_output=True, text=True, check=True
).stdout.strip()
ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")
so_path = os.path.join(build_dir, f"ambs_core{ext_suffix}")

cmd = [
    "g++", "-O3", "-Wall", "-shared", "-std=c++17", "-fPIC", "-fopenmp",
    f"-I{py_include}", f"-I{pybind_include}",
    os.path.join(build_dir, "ambs_core.cpp"),
    "-o", so_path,
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print("--- compiler stdout ---")
print(result.stdout)
print("--- compiler stderr ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Compilation failed (exit code {result.returncode}). See stderr above.")

if not os.path.exists(so_path):
    raise RuntimeError(f"Compiler reported success but {so_path} was not created.")

print(f"\nBuild OK -> {so_path}")

# Make sure the current directory is importable, then load the module.
if build_dir not in sys.path:
    sys.path.insert(0, build_dir)

import importlib
if "ambs_core" in sys.modules:
    importlib.reload(sys.modules["ambs_core"])
import ambs_core

print("ambs_core module loaded OK. Detected CPU cores:", os.cpu_count())


Building in: /content
Running: g++ -O3 -Wall -shared -std=c++17 -fPIC -fopenmp -I/usr/include/python3.13 -I/usr/local/lib/python3.13/dist-packages/pybind11/include /content/ambs_core.cpp -o /content/ambs_core.cpython-313-x86_64-linux-gnu.so
--- compiler stdout ---

--- compiler stderr ---


Build OK -> /content/ambs_core.cpython-313-x86_64-linux-gnu.so
ambs_core module loaded OK. Detected CPU cores: 2


In [ ]:
"""AMBS heuristic for the S-CLSP (Case I): port of the DynaPlex reference model (van Hezewijk et al., 2025,
Table 3).
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
from tqdm import tqdm

import ambs_core  # compiled C++ extension (see the two cells above)

# 0) Global configuration
SEED = 123
INSTANCE_ID = "1.8"


# 1) Instance registries (paper ID -> builder parameters)

# Case I: homogeneous demand, K in {5, ..., 30}, cf in {1.1, 1.5}, cov in {low, high} (same IDs as main.ipynb).
CASE_I_REGISTRY = {
    # K = 5
    "1.1":  dict(K=5,  cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.2":  dict(K=5,  cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.3":  dict(K=5,  cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.4":  dict(K=5,  cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 10
    "1.5":  dict(K=10, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.6":  dict(K=10, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.7":  dict(K=10, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.8":  dict(K=10, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 15
    "1.9":  dict(K=15, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.10": dict(K=15, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.11": dict(K=15, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.12": dict(K=15, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 20
    "1.13": dict(K=20, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.14": dict(K=20, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.15": dict(K=20, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.16": dict(K=20, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 25
    "1.17": dict(K=25, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.18": dict(K=25, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.19": dict(K=25, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.20": dict(K=25, cf=1.5, cov="high"),  # D_i = U{0,8}

    # K = 30
    "1.21": dict(K=30, cf=1.1, cov="low"),   # D_i = U{3,5}
    "1.22": dict(K=30, cf=1.5, cov="low"),   # D_i = U{3,5}
    "1.23": dict(K=30, cf=1.1, cov="high"),  # D_i = U{0,8}
    "1.24": dict(K=30, cf=1.5, cov="high"),  # D_i = U{0,8}
}


# 2) Instance model (mirrors DynaPlex's Paper1SettingsK{5,10,15}, no batch sizes)

@dataclass
class CLSPInstance:
    """No-batch-size S-CLSP instance (DynaPlex `mdp.cpp` instance model).

    :param K: Number of products.
    :param mu: Mean demand per product.
    :param sigma: DynaPlex "initial_deviation" (half-range for uniform demand).
    :param h: Holding cost per product.
    :param b: Backorder cost per product.
    :param k: Setup cost per product.
    :param theta: Setup time per product (int, capacity units).
    :param capacity: C-bar, per-period capacity (raw units, no batching).
    :param demand_corr_mode: Optional structured demand mode (not used by Case I): None | "pair_plus_one" |
        "pair_same" | "group_same".
    :param demand_corr_pairs: (i, j) index pairs for the "pair_*" modes.
    :param demand_corr_groups: Index groups for the "group_same" mode.
    """
    K: int
    mu: np.ndarray
    sigma: np.ndarray
    h: np.ndarray
    b: np.ndarray
    k: np.ndarray
    theta: np.ndarray
    capacity: int
    demand_corr_mode: Optional[str] = None
    demand_corr_pairs: Optional[list] = None
    demand_corr_groups: Optional[list] = None


def build_inst_set1(K: int, cov_level: str, cf: float,
                     mu_value: float = 4.0, h_value: float = 1.0,
                     b_value: float = 9.0, k_value: float = 200.0,
                     theta_value: int = 0) -> CLSPInstance:
    """Case I instance (Paper1SettingsK5/K10/K15 of mdp.cpp, homogeneous demand).

    penaltyCost = 9, holdingCost = 1, setupCost = 200, setupTime = 0, capacity = int(K * 4 * cf),
    initial_mean = 4, initial_deviation = 1.0 (U{3,5}) or 4.0 (U{0,8}).

    :param K: Number of products (5, 10, or 15 for the paper IDs used here).
    :param cov_level: "low" (-> U{3,5}, sigma=1.0) or "high" (-> U{0,8}, sigma=4.0).
    :param cf: Capacity factor (1.1 or 1.5).
    """
    assert cov_level in ("low", "high")
    sigma_value = 1.0 if cov_level == "low" else mu_value

    mu = np.full(K, mu_value, dtype=float)
    sigma = np.full(K, sigma_value, dtype=float)
    h = np.full(K, h_value, dtype=float)
    b = np.full(K, b_value, dtype=float)
    k = np.full(K, k_value, dtype=float)
    theta = np.full(K, theta_value, dtype=np.int64)

    # C++ does: static_cast<int64_t>(nr_products * 4 * capFactor) -> truncation, not rounding.
    capacity = int(K * mu_value * cf)

    return CLSPInstance(K=K, mu=mu, sigma=sigma, h=h, b=b, k=k, theta=theta, capacity=capacity)


def build_inst_from_instance_id(instance_id: str) -> Tuple[CLSPInstance, str]:
    instance_id = instance_id.strip()

    if instance_id in CASE_I_REGISTRY:
        p = CASE_I_REGISTRY[instance_id]
        inst = build_inst_set1(K=p["K"], cov_level=p["cov"], cf=p["cf"])
        exp_name = f"ID{instance_id}_K{p['K']}_cov{p['cov']}_cf{p['cf']}"
        return inst, exp_name

    raise ValueError(f"Unknown INSTANCE_ID '{instance_id}' for S-CLSP.")


# 3) Demand model - matches DynaPlex::DiscreteDist::GetUniformSample exactly

class DemandGenerator:
    """Discrete uniform demand per product, as GetUniformSample(mean, range): d ~ U{round(mean - range),
    ..., round(mean + range)}.

    Optional correlation modes (unused in Case I; no-op when `demand_corr_mode` is None):
    "pair_plus_one" / "pair_same" (per (i, j): d[j] = base + 1 / base, clipped), "group_same" (one base per
    group).
    """

    def __init__(self, inst: CLSPInstance):
        self.inst = inst
        self.lower = np.round(inst.mu - inst.sigma).astype(np.int64)
        self.upper = np.round(inst.mu + inst.sigma).astype(np.int64)
        self.lower = np.maximum(self.lower, 0)  # demand cannot be negative

    def sample(self, rng: np.random.Generator) -> np.ndarray:
        d = rng.integers(self.lower, self.upper + 1, dtype=np.int64)

        mode = self.inst.demand_corr_mode
        if mode == "pair_plus_one" and self.inst.demand_corr_pairs:
            for (i, j) in self.inst.demand_corr_pairs:
                base = rng.integers(self.lower[i], self.upper[i] + 1, dtype=np.int64)
                d[i] = base
                d[j] = np.clip(base + 1, self.lower[j], self.upper[j])

        elif mode == "pair_same" and self.inst.demand_corr_pairs:
            for (i, j) in self.inst.demand_corr_pairs:
                base = rng.integers(self.lower[i], self.upper[i] + 1, dtype=np.int64)
                d[i] = np.clip(base, self.lower[i], self.upper[i])
                d[j] = np.clip(base, self.lower[j], self.upper[j])

        elif mode == "group_same" and self.inst.demand_corr_groups:
            for grp in self.inst.demand_corr_groups:
                grp = list(grp)
                if not grp:
                    continue
                i0 = grp[0]
                base = rng.integers(self.lower[i0], self.upper[i0] + 1, dtype=np.int64)
                for j in grp:
                    d[j] = np.clip(base, self.lower[j], self.upper[j])

        return d

    def expected_backlog(self, product_index: int, inventory_plus_planned: float) -> float:
        """Expected backlog units E[max(d_i - x, 0)], not multiplied by b_i
        (MDP::CalculateExpectedBacklogCosts).
        """
        lo, hi = int(self.lower[product_index]), int(self.upper[product_index])
        n = hi - lo + 1
        total = 0.0
        p = 1.0 / n
        for d in range(lo, hi + 1):
            total += p * max(d - inventory_plus_planned, 0.0)
        return total


# 4) EOQ helpers

def eoq(mu: np.ndarray, k: np.ndarray, h: np.ndarray) -> np.ndarray:
    """Q_i = ceil(sqrt(2 * mu_i * k_i / h_i)), as in mdp.cpp."""
    return np.ceil(np.sqrt(2.0 * mu * k / h)).astype(np.int64)


def eoq_cost(mu: np.ndarray, k: np.ndarray, h: np.ndarray, Q: np.ndarray) -> np.ndarray:
    """EOQCost_i = k_i * mu_i / Q_i + h_i * Q_i / 2."""
    return k * mu / Q + h * Q / 2.0


def make_thresholds(inst: CLSPInstance, xb: float, xh: float, zmax) -> Tuple[float, float, int]:
    """AMBS thresholds from the scaling parameters: Bmin = xb * mean(EOQCost), Hmax = xh, Zmax = int(zmax)."""
    Q = eoq(inst.mu, inst.k, inst.h)
    C = eoq_cost(inst.mu, inst.k, inst.h, Q)
    Bmin = float(xb) * float(np.mean(C))
    Hmax = float(xh)
    return Bmin, Hmax, int(zmax)


# 5) AMBS parameters / result containers

@dataclass
class AMBSParams:
    Bmin: float
    Hmax: float
    Zmax: int


@dataclass
class AMBSResult:
    """Per-trajectory cost breakdown (only when `detailed=True`)."""
    avg_cost: float
    avg_inv_cost: float
    avg_bo_cost: float
    avg_setup_cost: float
    avg_setups: float
    avg_lot_size_units: float


# 6) Deterministic initial state (GetInitialState, mdp.cpp)

def initial_inventory(mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    """ceil(mu + 1.282 * sigma), deterministic (90% base stock)."""
    return np.ceil(mu + 1.282 * sigma).astype(np.int64)


# 7) The AMBS heuristic itself - faithful port of DetermineAMBSProductionPlan

class AMBSHeuristic:
    def __init__(self, inst: CLSPInstance):
        self.inst = inst
        self.Q = eoq(inst.mu, inst.k, inst.h)                 # EOQ_i
        self.C = eoq_cost(inst.mu, inst.k, inst.h, self.Q)    # EOQCost_i
        self.demand = DemandGenerator(inst)

    def determine_production_plan(self, inventory: np.ndarray, product_currently_set_up: int,
                                   used_capacity: int, params: AMBSParams
                                   ) -> Tuple[np.ndarray, int]:
        """Port of MDP::DetermineAMBSProductionPlan: builds the period's full plan one capacity unit at a
        time.

        Returns (planned_production[K], last_product_scheduled).
        """
        K = self.inst.K
        capacity = self.inst.capacity
        planned_capacity = int(used_capacity)
        planned_production = np.zeros(K, dtype=np.int64)
        new_setup_skus: List[int] = []
        last_product_determined = False
        last_product_scheduled: Optional[int] = None

        while planned_capacity < capacity:
            ebo = np.zeros(K, dtype=np.float64)
            ebo_exceeded = np.zeros(K, dtype=np.float64)

            for i in range(K):
                eligible = (
                    i in new_setup_skus
                    or i == product_currently_set_up
                    or len(new_setup_skus) < params.Zmax   # STRICT '<' (fixes the off-by-one bug)
                )
                if eligible:
                    ebo[i] = self.demand.expected_backlog(
                        i, float(inventory[i] + planned_production[i])
                    )
                # params.Bmin is already the scaled threshold (xb * mean EOQ cost): no rescaling by C[i].
                ebo_exceeded[i] = max(ebo[i] - params.Bmin, 0.0)

            if ebo_exceeded.max() > 0.0:
                # "STEP 1"-equivalent: prioritize the product most exceeding Bmin
                i_best = int(np.argmax(ebo_exceeded))

                if i_best not in new_setup_skus and i_best != product_currently_set_up:
                    new_setup_skus.append(i_best)
                    planned_capacity += int(self.inst.theta[i_best])
                    if planned_capacity >= capacity:
                        last_product_determined = True
                        last_product_scheduled = i_best

                if planned_capacity < capacity:
                    planned_production[i_best] += 1
                    planned_capacity += 1

            else:
                # No product exceeds the backorder threshold: fill capacity sensibly
                produced_skus = [product_currently_set_up] + new_setup_skus
                assert len(produced_skus) == len(set(produced_skus)), \
                    "current setup ended up duplicated in new_setups list"

                if len(produced_skus) <= 2:
                    last_product_determined = True
                    last_product_scheduled = produced_skus[0] if len(produced_skus) == 1 else produced_skus[1]

                    i0 = produced_skus[0]
                    while planned_capacity < capacity:
                        if inventory[i0] + planned_production[i0] + 1 < params.Hmax * self.Q[i0]:
                            planned_production[i0] += 1
                            planned_capacity += 1
                        else:
                            break
                else:
                    inv_ratio_new = [
                        (planned_production[s] + inventory[s]) / self.Q[s] for s in new_setup_skus
                    ]
                    last_product_scheduled = new_setup_skus[int(np.argmin(inv_ratio_new))]
                    last_product_determined = True
                    produced_skus.remove(last_product_scheduled)

                    inv_ratio = [
                        (planned_production[s] + inventory[s]) / self.Q[s] for s in produced_skus
                    ]
                    while planned_capacity < capacity and len(produced_skus) > 0:
                        j = int(np.argmin(inv_ratio))
                        s = produced_skus[j]
                        if inventory[s] + planned_production[s] + 1 < params.Hmax * self.Q[s]:
                            planned_production[s] += 1
                            planned_capacity += 1
                        else:
                            produced_skus.pop(j)
                            inv_ratio.pop(j)

                while planned_capacity < capacity and last_product_determined:
                    s = last_product_scheduled
                    if inventory[s] + planned_production[s] + 1 < params.Hmax * self.Q[s]:
                        planned_production[s] += 1
                        planned_capacity += 1
                    else:
                        planned_capacity = capacity  # force stop, mirrors mdp.cpp

        if not last_product_determined:
            if len(new_setup_skus) == 0:
                last_product_scheduled = product_currently_set_up
            else:
                inv_ratio_new = [
                    (planned_production[s] + inventory[s]) / self.Q[s] for s in new_setup_skus
                ]
                last_product_scheduled = new_setup_skus[int(np.argmin(inv_ratio_new))]

        return planned_production, int(last_product_scheduled)

    def realize_production_plan(self, planned_production: np.ndarray, product_currently_set_up: int,
                                 last_product_scheduled: int, used_capacity: int
                                 ) -> Tuple[np.ndarray, int, float, int]:
        """Execute the plan as AMBSPolicy::GetAction does: each product with planned quantity is produced
        once.

        Order: set-up product (unless it is the last one), remaining products by index, last scheduled
        product last.
        Quantities are capped by the remaining capacity at unit precision. A setup is counted (and charged
        inst.k[i])
        the first time a new product is produced.
        Returns (actual_production[K], n_new_setups, setup_cost_total, capacity_used).
        """
        K = self.inst.K
        capacity = self.inst.capacity

        order = []
        if planned_production[product_currently_set_up] > 0 and product_currently_set_up != last_product_scheduled:
            order.append(product_currently_set_up)
        for i in range(K):
            if i in (product_currently_set_up, last_product_scheduled):
                continue
            if planned_production[i] > 0:
                order.append(i)
        if planned_production[last_product_scheduled] > 0 or last_product_scheduled == product_currently_set_up:
            order.append(last_product_scheduled)

        actual = np.zeros(K, dtype=np.int64)
        capacity_used = int(used_capacity)
        n_new_setups = 0
        setup_cost_total = 0.0
        current_setup = product_currently_set_up

        for i in order:
            remaining = capacity - capacity_used
            if remaining <= 0:
                break
            needs_setup = (i != current_setup)
            if needs_setup:
                remaining -= int(self.inst.theta[i])
                if remaining <= 0:
                    continue
                n_new_setups += 1
                setup_cost_total += float(self.inst.k[i])
                capacity_used += int(self.inst.theta[i])
                current_setup = i

            produced_qty = min(int(planned_production[i]), capacity - capacity_used)
            if produced_qty <= 0:
                continue
            actual[i] = produced_qty
            capacity_used += produced_qty

        return actual, n_new_setups, setup_cost_total, capacity_used


# 8) Trajectory generator - SeedSequence-based, independent per trajectory

def get_trajectory_rng(base_seed: int, trajectory_index: int) -> np.random.Generator:
    seed_sequence = np.random.SeedSequence(base_seed, spawn_key=(trajectory_index,))
    return np.random.default_rng(seed_sequence)


# 9) One full trajectory: 128 warm-up + 100 measured periods (228 total)

def simulate_trajectory(inst: CLSPInstance, params: AMBSParams, rng: np.random.Generator,
                         warmup_periods: int = 128, periods_measured: int = 100,
                         detailed: bool = False):
    """One trajectory as in PolicyComparer::ComputeReturns: mean cost per period over `periods_measured`
    after `warmup_periods`.

    :param detailed: If True, returns an `AMBSResult` with the full cost breakdown
        (holding/backorder/setup/setups/units) instead of a single float. Purely additional bookkeeping;
        does not affect any production decision.
    """
    heuristic = AMBSHeuristic(inst)

    inventory = initial_inventory(inst.mu, inst.sigma)
    product_currently_set_up = 0     # deterministic, matches GetInitialState
    used_capacity = 0                # no setup carryover at t=0

    total_periods = warmup_periods + periods_measured
    cumulative_cost = 0.0
    checkpoint_at_warmup = 0.0

    if detailed:
        cum = dict(inv=0.0, bo=0.0, setup=0.0, setups=0.0, units=0.0)
        chk = dict(inv=0.0, bo=0.0, setup=0.0, setups=0.0, units=0.0)

    for t in range(total_periods):
        plan, last_product = heuristic.determine_production_plan(
            inventory, product_currently_set_up, used_capacity, params
        )
        actual_production, n_new_setups, setup_cost_period, _ = heuristic.realize_production_plan(
            plan, product_currently_set_up, last_product, used_capacity
        )

        inventory = inventory + actual_production
        demand_draw = heuristic.demand.sample(rng)
        inventory = inventory - demand_draw

        holding_cost = float(np.sum(inst.h * np.maximum(inventory, 0)))
        backorder_cost = float(np.sum(inst.b * np.maximum(-inventory, 0)))
        period_cost = holding_cost + backorder_cost + setup_cost_period

        cumulative_cost += period_cost
        if detailed:
            cum["inv"] += holding_cost
            cum["bo"] += backorder_cost
            cum["setup"] += setup_cost_period
            cum["setups"] += n_new_setups
            cum["units"] += float(np.sum(actual_production))

        product_currently_set_up = last_product
        used_capacity = 0  # theta carryover never triggers across periods here

        if t == warmup_periods - 1:
            checkpoint_at_warmup = cumulative_cost
            if detailed:
                chk = dict(cum)

    denom = float(periods_measured)
    avg_cost = (cumulative_cost - checkpoint_at_warmup) / denom
    if not detailed:
        return avg_cost

    return AMBSResult(
        avg_cost=avg_cost,
        avg_inv_cost=(cum["inv"] - chk["inv"]) / denom,
        avg_bo_cost=(cum["bo"] - chk["bo"]) / denom,
        avg_setup_cost=(cum["setup"] - chk["setup"]) / denom,
        avg_setups=(cum["setups"] - chk["setups"]) / denom,
        avg_lot_size_units=(cum["units"] - chk["units"]) / denom,
    )


# 10) Full test protocol: N trajectories, mean + standard error

def _inst_to_cpp_args(inst: CLSPInstance) -> dict:
    """Flatten a CLSPInstance into plain Python types for the pybind11 call."""
    return dict(
        K=inst.K,
        mu=inst.mu.tolist(), sigma=inst.sigma.tolist(),
        h=inst.h.tolist(), b=inst.b.tolist(), k=inst.k.tolist(),
        theta=[int(x) for x in inst.theta.tolist()],
        capacity=int(inst.capacity),
        corr_mode=inst.demand_corr_mode or "",
        corr_pairs=[tuple(p) for p in (inst.demand_corr_pairs or [])],
        corr_groups=[list(g) for g in (inst.demand_corr_groups or [])],
    )


def evaluate_ambs(inst: CLSPInstance, params: AMBSParams, n_trajectories: int = 10_000,
                   base_seed: int = 13_021_984, warmup_periods: int = 128,
                   periods_measured: int = 100, show_progress: bool = False,
                   per_product: bool = False) -> Tuple[float, float]:
    """Mean cost over `n_trajectories` independent trajectories, simulated in the C++ extension `ambs_core`.

    The C++ RNG (mt19937_64 per trajectory) is not bit-identical to numpy's, so results differ from a
    pure-Python run only statistically.
    `show_progress` is kept for API compatibility and has no effect.
    """
    args = _inst_to_cpp_args(inst)
    mean, se = ambs_core.evaluate(
        args['K'], args['mu'], args['sigma'], args['h'], args['b'], args['k'],
        args['theta'], args['capacity'], args['corr_mode'], args['corr_pairs'], args['corr_groups'],
        Bmin=params.Bmin, Hmax=params.Hmax, Zmax=params.Zmax,
        n_trajectories=n_trajectories, base_seed=base_seed,
        warmup_periods=warmup_periods, periods_measured=periods_measured,
        per_product=per_product,
    )
    return mean, se


# 11) Grid search for (Bmin, Hmax, Zmax)

def grid_search_ambs(inst: CLSPInstance, xb_grid=None, xh_grid=None, z_grid=None,
                      n_trajectories: int = 1000, base_seed: int = 10_000,
                      warmup_periods: int = 128, periods_measured: int = 100,
                      n_threads: int = 0) -> Tuple[AMBSParams, float]:
    """Grid search over (Bmin, Hmax, Zmax) built from (xb, xh, z); the whole grid is evaluated in C++
    (OpenMP over combinations).


    :param n_threads: OpenMP threads; 0 uses all cores.
    """
    if xb_grid is None:
        xb_grid = [i / 10.0 for i in range(0, 11)]
    if xh_grid is None:
        xh_grid = [i / 10.0 for i in range(5, 11)]
    if z_grid is None:
        z_grid = list(range(1, max(2, inst.K)))

    args = _inst_to_cpp_args(inst)
    res = ambs_core.grid_search(
        args['K'], args['mu'], args['sigma'], args['h'], args['b'], args['k'],
        args['theta'], args['capacity'], args['corr_mode'], args['corr_pairs'], args['corr_groups'],
        xb_grid=list(xb_grid), xh_grid=list(xh_grid), z_grid=list(z_grid),
        n_trajectories=n_trajectories, base_seed=base_seed,
        warmup_periods=warmup_periods, periods_measured=periods_measured,
        n_threads=n_threads,
    )
    best_params = AMBSParams(Bmin=res['Bmin'], Hmax=res['Hmax'], Zmax=res['Zmax'])
    return best_params, res['cost']


# 12) Instance construction according to paper INSTANCE_ID
inst, exp_name = build_inst_from_instance_id(INSTANCE_ID)

# 13) Execution: (i) threshold calibration; (ii) comparable evaluation
GRID_SEARCH_N_TRAJECTORIES = 1000
GRID_SEARCH_SEED_EVAL = 10_000
GRID_SEARCH_WARMUP_PERIODS = 128
GRID_SEARCH_PERIODS_MEASURED = 100

best_params, best_grid_cost = grid_search_ambs(
    inst,
    xb_grid=[i / 10.0 for i in range(0, 11)],
    xh_grid=[i / 10.0 for i in range(5, 11)],
    z_grid=list(range(1, max(2, inst.K))),
    n_trajectories=GRID_SEARCH_N_TRAJECTORIES,
    base_seed=GRID_SEARCH_SEED_EVAL,
    warmup_periods=GRID_SEARCH_WARMUP_PERIODS,
    periods_measured=GRID_SEARCH_PERIODS_MEASURED,
    n_threads=0,  # 0 = use all cores the runtime reports
)

print("\n==============================================")
print(f"AMBS experiment: {exp_name}")
print("== Best thresholds found (grid) ==")
print(f"Bmin = {best_params.Bmin:.4f}, Hmax = {best_params.Hmax:.4f}, Zmax = {best_params.Zmax}")
print(f"Cost (grid evaluation) = {best_grid_cost:.3f}")
print("==============================================\n")

# Individual runs (detailed breakdown, small diagnostic sample).
N_RUNS = 10
WARMUP_PERIODS = 128
PERIODS_MEASURED = 100
SEED_BASE_TEST = 12

run_costs, run_inv_costs, run_bo_costs, run_setup_costs = [], [], [], []
run_setups_rate, run_lot_units = [], []

print("== Individual AMBS results (SeedSequence-independent trajectories) ==")
for r in range(N_RUNS):
    rng_r = get_trajectory_rng(SEED_BASE_TEST, r)
    res_r = simulate_trajectory(inst, best_params, rng_r,
                                 warmup_periods=WARMUP_PERIODS,
                                 periods_measured=PERIODS_MEASURED,
                                 detailed=True)

    print(f"\n-- Run {r + 1}/{N_RUNS} --")
    print(f"Average total cost : {res_r.avg_cost:.3f}")
    print(f" - Inventory : {res_r.avg_inv_cost:.3f}")
    print(f" - Backorder : {res_r.avg_bo_cost:.3f}")
    print(f" - Setups    : {res_r.avg_setup_cost:.3f}")
    print(f"Setups / period : {res_r.avg_setups:.3f}")
    print(f"Avg units / period : {res_r.avg_lot_size_units:.3f}")

    run_costs.append(res_r.avg_cost)
    run_inv_costs.append(res_r.avg_inv_cost)
    run_bo_costs.append(res_r.avg_bo_cost)
    run_setup_costs.append(res_r.avg_setup_cost)
    run_setups_rate.append(res_r.avg_setups)
    run_lot_units.append(res_r.avg_lot_size_units)

print(f"\n== Average over {N_RUNS} runs (warmup={WARMUP_PERIODS}, measured={PERIODS_MEASURED}) ==")
print(f"Average total cost : {float(np.mean(run_costs)):.3f}")
print(f" - Inventory : {float(np.mean(run_inv_costs)):.3f}")
print(f" - Backorder : {float(np.mean(run_bo_costs)):.3f}")
print(f" - Setups    : {float(np.mean(run_setup_costs)):.3f}")
print(f"Setups / period : {float(np.mean(run_setups_rate)):.3f}")
print(f"Avg units / period : {float(np.mean(run_lot_units)):.3f}")
print("==============================================")

# Final evaluation: DynaPlex / Table-3 test protocol.
N_TRAJECTORIES = 10_000

mean_raw, _ = evaluate_ambs(inst, best_params, n_trajectories=N_TRAJECTORIES,
                                  base_seed=SEED_BASE_TEST, warmup_periods=WARMUP_PERIODS,
                                  periods_measured=PERIODS_MEASURED, show_progress=True)
mean_pp, _ = evaluate_ambs(inst, best_params, n_trajectories=N_TRAJECTORIES,
                                base_seed=SEED_BASE_TEST, warmup_periods=WARMUP_PERIODS,
                                periods_measured=PERIODS_MEASURED, per_product=True)

print("\n==============================================")
print(f"AMBS experiment: {exp_name}  ({N_TRAJECTORIES} trajectories, "
      f"{WARMUP_PERIODS}+{PERIODS_MEASURED} periods each)")
print(f"  raw (sum over products)  : mean={mean_raw:.3f}")
print("==============================================")
